In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [0]:
SOURCES = {
    "customers":                 "olist_customers_dataset.csv",
    "geolocation":               "olist_geolocation_dataset.csv",
    "order_items":               "olist_order_items_dataset.csv",
    "order_payments":            "olist_order_payments_dataset.csv",
    "order_reviews":             "olist_order_reviews_dataset.csv",
    "orders":                    "olist_orders_dataset.csv",
    "products":                  "olist_products_dataset.csv",
    "sellers":                   "olist_sellers_dataset.csv",
    "category_name_translation": "product_category_name_translation.csv",
}
path  = "olist_brazilian_ecommerce.bronze.{table_name}_raw"

In [0]:
customers =  spark.read.table("olist_brazilian_ecommerce.bronze.customers_raw")
orders =  spark.read.table("olist_brazilian_ecommerce.bronze.orders_raw")

In [0]:
unmatched_orders = (
    orders.alias("o")
    .join(
        customers.select("customer_id").alias("c"),
        "customer_id",
        "left_anti"
    )
)

print("Unmatched orders:", unmatched_orders.count())

Unmatched orders per  customers: 0 

In [0]:
customers_wihout_orders = (
    customers
    .join(
        orders.select("customer_id"),
        "customer_id",
        "left_anti"
    )
)

print("customers_wihout_orders", customers_wihout_orders.count())

In [0]:
order_reviews = spark.read.table("olist_brazilian_ecommerce.bronze.order_reviews_raw")

In [0]:
orders_without_review = (
    orders.alias("o")
    .join(
        order_reviews.select("order_id").alias("c"),
        "order_id",
        "left_anti"
    )
)

print("orders without review :", orders_without_review.count())

In [0]:
reviews_without_orders = (
    order_reviews.alias("r")
    .join(
        orders.select("order_id").alias("o"),
        "order_id",
        "left_anti"
    )
)

print("reviews without orders:", reviews_without_orders.count())

In [0]:
order_payments =  spark.read.table("olist_brazilian_ecommerce.bronze.order_payments_raw")

In [0]:
orders_without_payments  = (
    orders.alias("r")
    .join(
        order_payments.select("order_id").alias("p"),
        "order_id",
        "left_anti"
    )
)

print("orders without payments :", orders_without_payments.count())

In [0]:
orders_without_payments.display()


bfbd0f9bdef84302105ad712db648a6c don't has payment


In [0]:
payments_without_order  = (
    order_payments.alias("p")
    .join(
        orders.select("order_id").alias("r"),
        "order_id",
        "left_anti"
    )
)

print("payments without order :", payments_without_order.count())

In [0]:
order_items  = "olist_brazilian_ecommerce.bronze.order_items_raw" 
order_items = spark.read.table(order_items)

In [0]:
orders_without_order_items  = (
    orders.alias("r")
    .join(
        order_items.select("order_id").alias("i"),
        "order_id",
        "left_anti"
    )
)

print("orders_without_order_items :", orders_without_order_items.count())

In [0]:
orders_without_order_items.display()

In [0]:
orders_without_order_items.groupBy("order_status") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

In [0]:
orders_without_order_items \
    .filter(col("order_status").isin(
        "created", "invoiced", "shipped"
    )) \
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp"
    ) \
    .show(truncate=False)

In [0]:
suspicious_orders.select(
    "order_id",
    "order_status",
    "order_purchase_timestamp"
).join(
    order_payments.select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_value"
    ),
    on="order_id",
    how="left"
).orderBy("order_status", "order_id").show(truncate=False)

In [0]:
items_without_order  = (
    order_items.alias("r")
    .join(
        orders.select("order_id").alias("i"),
        "order_id",
        "left_anti"
    )
)

print("items_without_order :", items_without_order.count())

In [0]:

products =  spark.read.table("olist_brazilian_ecommerce.bronze.products_raw")

In [0]:
items_not_exist  = (
    order_items.alias("r")
    .join(
        products.select("product_id").alias("pro"),
        "product_id",
        "left_anti"
    )
)

print("item ordered and  not exist :", items_not_exist.count())

In [0]:
sellers =  spark.read.table("olist_brazilian_ecommerce.bronze.sellers_raw")

In [0]:
order_items_without_seller  = (
    order_items.alias("r_i")
    .join(
        sellers.select("seller_id").alias("sel"),
        "seller_id",
        "left_anti"
    )
)

print("order_items_without_seller:", order_items_without_seller.count())

In [0]:
orders_customers = orders.join(
    customers,
    on="customer_id",
    how="left"
)

print("Orders:", orders.count())
print("After join:", orders_customers.count())


## ## Relationship Testing Results

### 1. Orders ↔ Customers

**Tests performed:**
- Orders without matching customers.
- Customers without matching orders.
- Row count before and after a left join.

| Check | Result |
|---|---:|
| Unmatched orders | 0 |
| Customers without orders | 0 |
| Orders before join | 99,441 |
| Orders after join | 99,441 |

**Finding:** All orders have matching customer records, and every customer has at least one matching order. The join preserves the number of order rows.

### 2. Orders ↔ Order Reviews

| Check | Result |
|---|---:|
| Orders without reviews | 768 |
| Reviews without matching orders | 0 |

**Finding:** All review records reference existing orders. However, 768 orders have no matching review. This is not necessarily a data-quality issue because not every order must have a review.

### 3. Orders ↔ Payments

| Check | Result |
|---|---:|
| Orders without payments | 1 |
| Payments without matching orders | 0 |

**Finding:** All payment records reference existing orders. One order has no matching payment.

**Order requiring investigation:** `bfbd0f9bdef84302105ad712db648a6c`

### 4. Orders ↔ Order Items

| Check | Result |
|---|---:|
| Orders without order items | 775 |
| Order items without matching orders | 0 |

The 775 orders without order items are distributed by status as follows:

| Order Status | Count |
|---|---:|
| unavailable | 603 |
| canceled | 164 |
| created | 5 |
| invoiced | 2 |
| shipped | 1 |
| **Total** | **775** |

**Finding:** Most orders without items have `unavailable` or `canceled` status. The five `created` orders, two `invoiced` orders, and one `shipped` order require further investigation. Some of these orders have payment records despite having no associated order items.

### 5. Order Items ↔ Products

| Check | Result |
|---|---:|
| Order items referencing missing products | 0 |

**Finding:** Every order item references an existing product.

### 6. Order Items ↔ Sellers

| Check | Result |
|---|---:|
| Order items referencing missing sellers | 0 |

**Finding:** Every order item references an existing seller.

### Overall Conclusion

The tested relationships show no unmatched records for orders and customers, reviews and orders, payments and orders in the reverse direction, order items and orders, order items and products, or order items and sellers.

The main findings requiring further investigation are:
- 768 orders without reviews.
- One order without a payment.
- 775 orders without order items, including eight with `created`, `invoiced`, or `shipped` status.

These results describe the source data and do not automatically determine which records should be removed or transformed in the Silver layer. Further investigation is required before defining cleaning rules.